# JetRacer: Road Following + Deteção de Sinais STOP / Perigo

Este notebook estende a lógica do `road_following.ipynb` oficial do JetRacer.

Ideia geral:

```text
câmara CSI
   ↓
modelo road following → steering
   ↓
detetor OpenCV de sinais → decisão sobre throttle
   ↓
JetRacer: direção + velocidade
```

O modelo de road following continua responsável pela direção.  
A deteção de sinais apenas altera a velocidade:

- **STOP**: parar durante alguns segundos.
- **PERIGO / TRIÂNGULO**: abrandar durante alguns segundos.

> Antes de correr com o robô no chão, levantar as rodas ou colocar o carro num suporte.

## 1. Imports

In [ ]:
import time
import cv2
import numpy as np

import torch
import torchvision

from torch2trt import TRTModule
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar
from utils import preprocess

## 2. Carregar modelo TensorRT de road following

In [ ]:
# Caminho do modelo TensorRT gerado a partir do road_following.ipynb oficial
MODEL_TRT_PATH = 'road_following_model_trt.pth'

CATEGORIES = ['apex']

model_trt = TRTModule()
model_trt.load_state_dict(torch.load(MODEL_TRT_PATH))

print("Modelo TensorRT carregado:", MODEL_TRT_PATH)

## 3. Inicializar o JetRacer e a câmara

In [ ]:
car = NvidiaRacecar()

# 224x224 é o tamanho esperado pelo modelo de road following
camera = CSICamera(
    width=224,
    height=224,
    capture_width=1280,
    capture_height=720,
    capture_fps=30
)

print("JetRacer e câmara inicializados.")

## 4. Parâmetros de condução

Ajustar estes valores com cuidado.

- `STEERING_GAIN`: quanto a previsão da rede afeta a direção.
- `STEERING_BIAS`: correção de enviesamento, caso o carro tenda a ir para um lado.
- `THROTTLE_NORMAL`: velocidade normal.
- `THROTTLE_SLOW`: velocidade ao detetar sinal de perigo.
- `STOP_DURATION`: tempo parado após detetar STOP.

In [ ]:
STEERING_GAIN = 0.75
STEERING_BIAS = 0.00

THROTTLE_NORMAL = 0.15
THROTTLE_SLOW = 0.07
THROTTLE_STOP = 0.00

STOP_DURATION = 3.0     # segundos parado após STOP
SLOW_DURATION = 2.0     # segundos a abrandar após perigo

# Limites de segurança para steering
STEERING_MIN = -1.0
STEERING_MAX = 1.0

## 5. Função de deteção geométrica de sinais

Esta versão é propositadamente simples e didática:

- Converte a imagem para HSV.
- Segmenta zonas vermelhas.
- Procura contornos.
- Aproxima cada contorno por um polígono.
- Triângulo → `PERIGO`.
- Octógono aproximado → `STOP`.

Nota: esta abordagem é útil para demonstração, mas pode gerar falsos positivos. Para robustez, usar também tamanho, posição na imagem, cor, e eventualmente uma CNN treinada para sinais.

In [ ]:
def detectar_sinal(frame_rgb, debug=False):
    """Deteta sinais simples no frame RGB.

    Retorna:
        'STOP', 'PERIGO' ou None
    """

    # Trabalhar numa cópia para evitar alterar a imagem original usada pelo modelo
    frame = frame_rgb.copy()

    # CSICamera/JetCam costuma devolver RGB
    hsv = cv2.cvtColor(frame, cv2.COLOR_RGB2HSV)

    # Vermelho em HSV aparece em duas zonas de hue: perto de 0 e perto de 179
    lower_red1 = np.array([0, 80, 70])
    upper_red1 = np.array([10, 255, 255])

    lower_red2 = np.array([170, 80, 70])
    upper_red2 = np.array([179, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = cv2.bitwise_or(mask1, mask2)

    # Limpeza simples da máscara
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    melhor_sinal = None
    maior_area = 0

    for c in contours:
        area = cv2.contourArea(c)

        # Ajustar conforme tamanho dos sinais e distância à câmara
        if area < 80:
            continue

        perimeter = cv2.arcLength(c, True)

        if perimeter <= 0:
            continue

        approx = cv2.approxPolyDP(c, 0.04 * perimeter, True)
        sides = len(approx)

        x, y, w, h = cv2.boundingRect(approx)
        aspect = w / float(h) if h > 0 else 0

        sinal = None

        # Sinal triangular de perigo
        if sides == 3:
            sinal = 'PERIGO'

        # STOP: octógono. Usamos intervalo para tolerar ruído.
        elif 7 <= sides <= 9 and 0.65 <= aspect <= 1.35:
            sinal = 'STOP'

        if sinal is not None and area > maior_area:
            maior_area = area
            melhor_sinal = sinal

    return melhor_sinal

## 6. Função opcional para desenhar diagnóstico

Esta função permite visualizar no notebook qual sinal foi detetado.  
Não é necessária para condução autónoma, mas ajuda muito durante a afinação.

In [ ]:
def desenhar_diagnostico(frame_rgb, sinal, steering, throttle):
    frame = frame_rgb.copy()

    texto = "SINAL: {}".format(sinal if sinal is not None else "nenhum")
    cv2.putText(frame, texto, (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv2.putText(frame, "steering={:.2f} throttle={:.2f}".format(steering, throttle),
                (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1)

    return frame

## 7. Teste estático da câmara e do detetor

Correr esta célula antes de colocar o carro em movimento.

In [ ]:
from IPython.display import display
from PIL import Image

image = camera.read()
sinal = detectar_sinal(image)

print("Sinal detetado:", sinal)
display(Image.fromarray(desenhar_diagnostico(image, sinal, 0.0, 0.0)))

## 8. Ciclo principal: road following + sinais

Este ciclo faz:

1. Lê imagem da câmara.
2. Usa o modelo TensorRT para prever o apex.
3. Calcula direção.
4. Deteta sinal.
5. Aplica regra de decisão:
   - STOP → `throttle = 0` durante `STOP_DURATION`.
   - PERIGO → `throttle = THROTTLE_SLOW` durante `SLOW_DURATION`.
   - Caso contrário → `THROTTLE_NORMAL`.

Parar com `Kernel → Interrupt` ou `Ctrl+C` se estiver num terminal.

In [ ]:
stop_until = 0.0
slow_until = 0.0

car.throttle = 0.0
car.steering = 0.0

print("A iniciar. Interromper o kernel para parar.")

try:
    while True:
        image = camera.read()

        # --- Road following: direção pela CNN ---
        image_tensor = preprocess(image).half()
        output = model_trt(image_tensor).detach().cpu().numpy().flatten()

        x = float(output[0])
        steering = x * STEERING_GAIN + STEERING_BIAS
        steering = max(STEERING_MIN, min(STEERING_MAX, steering))

        # --- Deteção de sinais ---
        sinal = detectar_sinal(image)
        agora = time.time()

        if sinal == 'STOP':
            stop_until = agora + STOP_DURATION
            print("STOP detetado. A parar.", end='\r')

        elif sinal == 'PERIGO':
            slow_until = agora + SLOW_DURATION
            print("PERIGO detetado. A abrandar.", end='\r')

        # --- Decisão final de velocidade ---
        if agora < stop_until:
            throttle = THROTTLE_STOP

        elif agora < slow_until:
            throttle = THROTTLE_SLOW

        else:
            throttle = THROTTLE_NORMAL

        # --- Aplicar comandos ao carro ---
        car.steering = steering
        car.throttle = throttle

except KeyboardInterrupt:
    print("\nInterrompido pelo utilizador.")

finally:
    car.throttle = 0.0
    car.steering = 0.0
    print("JetRacer parado.")

## 9. Versão com visualização no Jupyter

Esta versão é mais lenta, mas útil para demonstração.  
Evitar usar a visualização se o objetivo for condução mais fluida.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

image_widget = widgets.Image(format='jpeg', width=300, height=300)
display(image_widget)

stop_until = 0.0
slow_until = 0.0

car.throttle = 0.0
car.steering = 0.0

try:
    while True:
        image = camera.read()

        image_tensor = preprocess(image).half()
        output = model_trt(image_tensor).detach().cpu().numpy().flatten()

        x = float(output[0])
        steering = x * STEERING_GAIN + STEERING_BIAS
        steering = max(STEERING_MIN, min(STEERING_MAX, steering))

        sinal = detectar_sinal(image)
        agora = time.time()

        if sinal == 'STOP':
            stop_until = agora + STOP_DURATION

        elif sinal == 'PERIGO':
            slow_until = agora + SLOW_DURATION

        if agora < stop_until:
            throttle = THROTTLE_STOP
        elif agora < slow_until:
            throttle = THROTTLE_SLOW
        else:
            throttle = THROTTLE_NORMAL

        car.steering = steering
        car.throttle = throttle

        diag = desenhar_diagnostico(image, sinal, steering, throttle)

        # imencode espera BGR; converter de RGB para BGR
        diag_bgr = cv2.cvtColor(diag, cv2.COLOR_RGB2BGR)
        _, jpeg = cv2.imencode('.jpg', diag_bgr)
        image_widget.value = jpeg.tobytes()

except KeyboardInterrupt:
    print("Interrompido.")

finally:
    car.throttle = 0.0
    car.steering = 0.0
    print("JetRacer parado.")

## 10. Notas para afinação

Se o robô parar sem haver STOP:

- aumentar `area < 80` para `area < 150` ou `area < 300`;
- tornar o critério do octógono mais exigente;
- exigir que o sinal apareça em várias frames consecutivas.

Se não detetar o STOP:

- aproximar o sinal;
- melhorar iluminação;
- ajustar os intervalos HSV;
- aumentar resolução da câmara;
- usar sinais maiores.

Versão robusta recomendada:

```text
road following CNN para condução
+
pequena CNN/classificador YOLO/SSD para sinais
+
máquina de estados para STOP, SLOW, NORMAL
```